In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Working")

c:\Users\KIIT0001\medical_chatbot-\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working


In [27]:
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_pinecone import PineconeVectorStore

from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from rank_bm25 import BM25Okapi

In [12]:
load_dotenv()

True

In [13]:
loader = PyPDFLoader("C:/Users/KIIT0001/medical_chatbot-/data/Medical_book.pdf")

documents = loader.load()

len(documents)

637

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks = splitter.split_documents(documents)

len(chunks)

9862

In [15]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [16]:
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding,
    index_name=os.getenv("PINECONE_INDEX"),
    batch_size=20
)

In [17]:
docs = vectorstore.similarity_search(
    "What are symptoms of diabetes?",
    k=3
)

for doc in docs:
    print(doc.page_content)
    print("=" * 50)

• Type I diabetes mellitus. Characterized by fatigue and
an abnormally high level of glucose in the blood
(hyperglycemia).
• Amyotrophic lateral schlerosis. First signs are stum-
bling and difficulty climbing stairs. Later, muscle
cramps and twitching may be observed as well as
begin to fall. A person with diabetes mellitus either does
not make enough insulin, or makes insulin that does not
work properly. The result is blood sugar that remains
high, a condition called hyperglycemia.
Diabetes must be diagnosed as early as possible. If
• heavy sweating
• oily skin
• increased coarse body hair
• improper processing of sugars in the diet (and some-
times actual diabetes)
• high blood pressure
• increased calcium in the urine (sometimes leading to
kidney stones)
• increased risk of gallstones; and
• swelling of the thyroid gland


In [18]:
docs = vectorstore.max_marginal_relevance_search(
    "What are symptoms of diabetes?",
    k=5,
    fetch_k=20
)

for doc in docs:
    print(doc.page_content)
    print("=" * 50)

• Type I diabetes mellitus. Characterized by fatigue and
an abnormally high level of glucose in the blood
(hyperglycemia).
• Amyotrophic lateral schlerosis. First signs are stum-
bling and difficulty climbing stairs. Later, muscle
cramps and twitching may be observed as well as
• heavy sweating
• oily skin
• increased coarse body hair
• improper processing of sugars in the diet (and some-
times actual diabetes)
• high blood pressure
• increased calcium in the urine (sometimes leading to
kidney stones)
• increased risk of gallstones; and
• swelling of the thyroid gland
1660 Duke Street, Alexandria, V A 22314. (800)232-3472.
<http://www.diabetes.org>.
GALE ENCYCLOPEDIA OF MEDICINE 2 263
Antidiabetic drugs
Antidiabetic Drugs
Brand Name(Generic Name) Possible Common Side Effects Include:
Diabinese (chlorpropamide) Diarrhea, nausea, loss of appetite
in their arms. Early symptoms include decrease in the
blood supply (arterial ischemia) and superficial (near the
skin surface) phlebitis. The m

In [19]:
corpus = [doc.page_content for doc in docs]

tokenized_corpus = [
    doc.split() for doc in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

scores = bm25.get_scores(
    "diabetes symptoms".split()
)

scores

array([1.11663874, 0.        , 0.        , 0.32851711, 0.32531242])

In [21]:
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant"
)

hyde_prompt = ChatPromptTemplate.from_template(
    """
    Write a detailed medical answer for:
    
    {question}
    """
)

hyde_chain = hyde_prompt | llm

response = hyde_chain.invoke({
    "question": "What causes diabetes?"
})

print(response.content)

**Understanding the Causes of Diabetes**

Diabetes is a complex and multifactorial disease that affects millions of people worldwide. It is a chronic condition characterized by high blood sugar levels, which can lead to various complications if left untreated or poorly managed. To comprehend the causes of diabetes, it is essential to delve into its underlying pathophysiology and the factors that contribute to its development.

**Types of Diabetes**

There are several types of diabetes, each with distinct causes and risk factors:

1. **Type 1 Diabetes (T1D)**: An autoimmune disease in which the body's immune system mistakenly attacks and destroys the insulin-producing beta cells in the pancreas. This results in a complete deficiency of insulin production, requiring patients to rely on exogenous insulin therapy.
2. **Type 2 Diabetes (T2D)**: The most common form of diabetes, accounting for approximately 90% of cases. T2D is characterized by insulin resistance, where the body's cells beco

In [22]:
decompose_prompt = ChatPromptTemplate.from_template(
    """
    Break the medical query into smaller sub-queries.
    
    Question:
    {question}
    """
)

decompose_chain = decompose_prompt | llm

response = decompose_chain.invoke({
    "question": "What causes diabetes and how is it treated?"
})

print(response.content)

To break down the medical query into smaller sub-queries, we can divide it into two main parts:

1. **What causes diabetes?**
   a. **What are the main types of diabetes?**
   b. **What are the risk factors for developing diabetes?**
   c. **How do lifestyle and genetics contribute to the development of diabetes?**

2. **How is diabetes treated?**
   a. **What are the main treatment options for diabetes?**
   b. **What role does medication play in managing diabetes?**
   c. **How do lifestyle changes and diet affect diabetes management?**

Here's a more detailed breakdown of the sub-queries:

**What causes diabetes?**

1. **What are the main types of diabetes?**
   * What are the differences between Type 1 and Type 2 diabetes?
   * What is Gestational diabetes, and how is it related to pregnancy?
   * What about LADA (Latent Autoimmune Diabetes in Adults) and MODY (Maturity-Onset Diabetes of the Young)?

2. **What are the risk factors for developing diabetes?**
   * What are the geneti

In [23]:
def hybrid_retrieve(query):

    dense_docs = vectorstore.max_marginal_relevance_search(
        query,
        k=5,
        fetch_k=20
    )

    corpus = [doc.page_content for doc in dense_docs]

    tokenized = [doc.split() for doc in corpus]

    bm25 = BM25Okapi(tokenized)

    scores = bm25.get_scores(query.split())

    ranked_docs = sorted(
        zip(scores, dense_docs),
        reverse=True,
        key=lambda x: x[0]
    )

    final_docs = [doc for _, doc in ranked_docs]

    return final_docs[:5]

In [28]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_12308\515166408.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [29]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a conversational medical AI assistant.

    Use only the provided context.

    Chat History:
    {chat_history}

    Context:
    {context}

    Question:
    {question}
    """
)

parser = StrOutputParser()

chain = prompt | llm | parser

In [32]:
query = "What are symptoms of diabetes?"

docs = hybrid_retrieve(query)

context = "\n".join([
    doc.page_content for doc in docs
])

# Load conversation memory
chat_history = memory.load_memory_variables({})[
    "chat_history"
]

response = chain.invoke({
    "context": context,
    "question": query,
    "chat_history": chat_history
})

# Save conversation
memory.save_context(
    {"input": query},
    {"output": response}
)

print(response)

Based on the provided context, the symptoms of diabetes, specifically Type I diabetes mellitus, include:

1. Fatigue
2. Hyperglycemia (an abnormally high level of glucose in the blood)

Please note that the context also mentions other conditions and symptoms, but the question specifically asks about symptoms of diabetes. If you'd like more information on other conditions, feel free to ask.


In [33]:
query = "How is it different from malaria?"

docs = hybrid_retrieve(query)

context = "\n".join([
    doc.page_content for doc in docs
])

chat_history = memory.load_memory_variables({})[
    "chat_history"
]

response = chain.invoke({
    "context": context,
    "question": query,
    "chat_history": chat_history
})

memory.save_context(
    {"input": query},
    {"output": response}
)

print(response)

Based on the provided context, it appears that the conversation started with discussing diabetes, but it seems you've asked a new question that diverges from the topic. However, I can provide information on the differences between diabetes and malaria.

Diabetes and malaria are two distinct medical conditions with different causes, symptoms, and treatments.

Diabetes is a metabolic disorder characterized by high blood sugar levels, which can be caused by several factors, including genetics, lifestyle, and environmental factors. The symptoms of diabetes include fatigue, hyperglycemia, and other complications.

Malaria, on the other hand, is a mosquito-borne infectious disease caused by the Plasmodium parasite. The symptoms of malaria include fever, chills, flu-like symptoms, and in severe cases, blackwater fever, which is a serious complication of one type of malaria.

Key differences between diabetes and malaria include:

1. **Cause**: Diabetes is caused by high blood sugar levels, whe